In [1]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = 'sqlite:///mlflow.db'
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [6]:
client.search_experiments()

[<Experiment: artifact_location='file:///c:/Users/AvadhootHublikar/Documents/mlops/mlops-zoomcamp/03-model_training/experiment_tracking/mlruns/3', creation_time=1774341698795, experiment_id='3', last_update_time=1774341698795, lifecycle_stage='active', name='demo-taxi-experiment', tags={}, workspace='default'>,
 <Experiment: artifact_location='file:///c:/Users/AvadhootHublikar/Documents/mlops/mlops-zoomcamp/03-model_training/experiment_tracking/mlruns/2', creation_time=1774341460975, experiment_id='2', last_update_time=1774341460975, lifecycle_stage='active', name='taxi-experiment', tags={}, workspace='default'>,
 <Experiment: artifact_location='file:///c:/Users/AvadhootHublikar/Documents/mlops/mlops-zoomcamp/03-model_training/experiment_tracking/mlruns/1', creation_time=1774340952030, experiment_id='1', last_update_time=1774340952030, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, workspace='default'>,
 <Experiment: artifact_location='file:///c:/Users/AvadhootHublikar/

In [7]:
client.create_experiment('my_cool_experiment')

'4'

In [50]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='3', 
    filter_string='', 
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=['metrics.rmse ASC']
)

In [51]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")


run id: e6bb9faf57ea478f992fb8d435340708, rmse: 6.8399
run id: 6062042323af4505b2a4f63c9625bd15, rmse: 6.8429
run id: f055a2e8f7b44b54b4c89e39116a156a, rmse: 6.8430
run id: cfede717916641eda7e1fe6474b324f2, rmse: 6.8447
run id: 8cbae70168a64d27bb5285b392b94148, rmse: 6.8478


In [52]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [ ]:
run_id = "449cb24233be460f8cc64ba31ca2147b"
model_uri = f"runs:/{run_id}/model"
#model = mlflow.xgboost.load_model(model_uri)
mlflow.register_model(model_uri=model_uri, name='nyc-taxi-regressor')

MlflowException: Failed to download artifacts from path 'model', please ensure that the path is correct.

In [67]:
model_name = 'nyc-taxi-regressor'
latest_version = client.get_latest_versions(name=model_name)

for version in latest_version:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 2, stage: Staging
version: 3, stage: None


C:\Users\AvadhootHublikar\AppData\Local\Temp\ipykernel_30784\470327430.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_version = client.get_latest_versions(name=model_name)


In [ ]:
model_version = 3
new_stage = 'Acceptance'

client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

C:\Users\AvadhootHublikar\AppData\Local\Temp\ipykernel_30784\3054706055.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1774547977664, current_stage='Staging', deployment_job_state=None, description='This model version 3 was transitioned to Acceptance on 2026-03-26.', last_updated_timestamp=1774549424760, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='449cb24233be460f8cc64ba31ca2147b', run_link=None, source='models:/m-603ad6132cfb4bd88dc0632d3518ac4a', status='READY', status_message=None, tags={}, user_id=None, version=3, workspace='default'>

In [75]:
from datetime import datetime
date = datetime.today().date()


client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"This model version {model_version} was transitioned to {new_stage} on {date}."
)

<ModelVersion: aliases=[], creation_timestamp=1774547977664, current_stage='Staging', deployment_job_state=None, description='This model version 3 was transitioned to Acceptance on 2026-03-26.', last_updated_timestamp=1774549428885, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='449cb24233be460f8cc64ba31ca2147b', run_link=None, source='models:/m-603ad6132cfb4bd88dc0632d3518ac4a', status='READY', status_message=None, tags={}, user_id=None, version=3, workspace='default'>

In [80]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd

def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)


    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration'] = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)

def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [81]:
df = read_dataframe('./data/green_tripdata_2021-03.parquet')

In [82]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'c:\\Users\\AvadhootHublikar\\Documents\\mlops\\mlops-zoomcamp\\03-model_training\\experiment_tracking\\preprocessor'

In [83]:
import pickle

with open('preprocessor/preprocessor.b', 'rb') as f_in:
    dv = pickle.load(f_in)

In [84]:
X_test = preprocess(df, dv)

In [85]:
target = 'duration'
y_test = df[target].values

In [86]:
%time test_model(name=model_name, stage='Production', X_test=X_test, y_test=y_test)

CPU times: total: 2.92 s
Wall time: 1.09 s


{'rmse': 7.795496599696125}

In [87]:
%time test_model(name=model_name, stage='Staging', X_test=X_test, y_test=y_test)

CPU times: total: 656 ms
Wall time: 72.9 ms


{'rmse': 7.795496599696125}

In [90]:
client.transition_model_version_stage(
    name=model_name,
    version=3,
    stage='Production',
    archive_existing_versions=True
)

C:\Users\AvadhootHublikar\AppData\Local\Temp\ipykernel_30784\3339164012.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1774547977664, current_stage='Production', deployment_job_state=None, description='This model version 3 was transitioned to Acceptance on 2026-03-26.', last_updated_timestamp=1774598352044, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='449cb24233be460f8cc64ba31ca2147b', run_link=None, source='models:/m-603ad6132cfb4bd88dc0632d3518ac4a', status='READY', status_message=None, tags={}, user_id=None, version=3, workspace='default'>